# Retail-store shoplifting: model development
**CSE 6367 Computer Vision** · Harsha Prasad and Himanshu Shimpi

Run cells in order using **Python 3.11 (Shoplifting)**. Each stage records its inputs, measured outputs and validation checks.

The workflow covers retail-video preparation, pose tracking, quality review, temporal preprocessing, feature construction, frozen MotionBERT lifting, matched A–E experiments, expanded three-seed validation and offline video inference.

Sections **10b–10c** preserve the original A/B measurements. Sections **10d–10f** inspect the versioned quality-corrected pipeline. The original and new runs use different preprocessing and calibration protocols; their difference is not a single-factor ablation.

Video labels do not identify the responsible person or exact theft interval. A skeleton or above-threshold reconstruction error is not proof of theft. Generated data and model artifacts stay in ignored `outputs/`.


## 0 · Environment and execution controls
Reuse verified outputs to avoid unnecessary inference. Set `RUN_POSE_INFERENCE=True` to compute missing pose exports. Enable training explicitly when a new run is needed; prerequisite checks remain active.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, hashlib
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("MKL_NUM_THREADS", "2")
from importlib.metadata import version
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import yaml
from IPython.display import display, Markdown, Video

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "scripts/extract_video_poses.py").is_file() and (p / "src").is_dir()), None)
assert ROOT is not None, "Open this notebook inside the project repository."
assert sys.version_info[:2] == (3, 11), "Select the project's Python 3.11 kernel."
assert Path(sys.prefix).resolve() == (ROOT / ".venv").resolve(), "Select Python 3.11 (Shoplifting), the project .venv kernel."
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ["YOLO_CONFIG_DIR"] = str(ROOT / "outputs/yolo_settings")
OUT = ROOT / "outputs/model_development"
OUT.mkdir(parents=True, exist_ok=True)

RUN_FRAME_EXTRACTION = False
RUN_BATCH_EXPORT = False
RUN_POSE_INFERENCE = False
RUN_RETAILS_EDA = False
RUN_TRAINING = False
RUN_3D_LIFT = False
RUN_VIDEO_INFERENCE = False
RESEARCH_STAGE = "matched"

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

def run_cli(*args):
    command = [sys.executable, *map(str, args)]
    result = subprocess.run(command, cwd=ROOT, capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()
    return result

print("Python:", sys.version.split()[0], "|", sys.executable)
print("Repository:", ROOT)
display(pd.DataFrame([{"package": name, "version": version(name)}
                     for name in ["torch", "ultralytics", "numpy", "opencv-python", "notebook"]]))
print("CHECKPOINT PASS: project Python 3.11 kernel selected.")

## 1 · Verify the retail-video inputs
the small UCF-Crime subset is external inspection data. RetailS is the separate dataset expected by the training code. Check metadata directly rather than relying on an old README result.

In [ ]:
setup_result = run_cli("scripts/check_project_setup.py")
setup = json.loads(setup_result.stdout)
rows = []
for label, group in setup["classes"].items():
    for video in group["videos"]:
        rows.append({"class": label, **video, "seconds": round(video["frames"] / video["fps"], 2)})
inventory = pd.DataFrame(rows)
assert inventory["readable"].all()
display(inventory)
print("CHECKPOINT PASS:", len(inventory), "readable retail videos.")
print("Later-stage prerequisites:", setup["later_stage_assets"])

## 2 · Inspect retail scenes and source videos
distinguish the store environment from a video-level event label. The selected Shoplifting clip is a clothing store; the Normal clip is an electronics store. These are genuine decoded frames, not generated images.

The helper transcodes short preview excerpts to browser-compatible H.264. This changes analysis copies only; inference continues to use the original videos.

In [ ]:
SOURCES = {
    "Shoplifting (dataset label)": ROOT / "data/external_video/videos/shoplifting/Shoplifting033_x264.mp4",
    "Normal (dataset label)": ROOT / "data/external_video/videos/normal/Normal_Videos_100_x264.mp4",
}

def read_frame(path, index):
    cap = cv2.VideoCapture(str(path))
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(index))
    ok, frame = cap.read()
    cap.release()
    assert ok, f"Cannot decode {path.name}, frame {index}"
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

def show_clip(path, start=0, seconds=8):
    import imageio_ffmpeg
    path = Path(path)
    key = hashlib.sha256(f"{path.resolve()}:{path.stat().st_mtime_ns}:{start}:{seconds}".encode()).hexdigest()[:12]
    preview = OUT / f"clip_{key}.mp4"
    if not preview.exists():
        command = [imageio_ffmpeg.get_ffmpeg_exe(), "-hide_banner", "-loglevel", "error", "-y",
                   "-ss", str(start), "-i", str(path), "-t", str(seconds), "-an",
                   "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "22",
                   "-movflags", "+faststart", str(preview)]
        subprocess.run(command, check=True, capture_output=True)
    cap = cv2.VideoCapture(str(preview))
    ok, _ = cap.read()
    cap.release()
    assert ok, "Video excerpt failed decoding"
    display(Video(filename=str(preview), embed=True, width=480))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (label, path) in zip(axes, SOURCES.items()):
    ax.imshow(read_frame(path, 200)); ax.set_title(label); ax.axis("off")
plt.tight_layout(); plt.show()
for label, path in SOURCES.items():
    display(Markdown(f"**{label}: `{path.name}`**"))
    show_clip(path, start=20 if "Shoplifting" in label else 0, seconds=8)

## 3 · Frame extraction and a chronological event excerpt
sparse JPEGs help inspect scenes. Temporal models require consecutive original video frames. The existing extraction script samples across each class; its JPEG numbers are not event timestamps.

The six frames below are decoded directly from **one video** at known frame IDs. They show movement near the display and toward a pocket, followed by movement away. The small item's visibility is limited; do not claim the images alone prove theft.

In [ ]:
if RUN_FRAME_EXTRACTION:
    print("Regenerating the 300 sampled JPEGs per class using the existing script.")
    run_cli("scripts/extract_frames.py", "--frames-per-class", "300")
frame_rows = []
for label in ["shoplifting", "normal"]:
    images = sorted((ROOT / "data/external_video/frames" / label).glob("*.jpg"))
    assert images and all(cv2.imread(str(p)) is not None for p in images), f"Invalid images in {label}"
    frame_rows.append({"class": label, "readable_JPEGs": len(images)})
display(pd.DataFrame(frame_rows))
indices = [660, 690, 705, 735, 750, 810]
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, index in zip(axes.flat, indices):
    ax.imshow(read_frame(SOURCES["Shoplifting (dataset label)"], index))
    ax.set_title(f"Frame {index} | {index/30:.1f} s"); ax.axis("off")
plt.tight_layout(); plt.show()
print("CHECKPOINT PASS: JPEGs decode; sequence frame IDs are ordered and from one source video.")

## 4 · YOLO Pose + ByteTrack: run or verify
a pretrained detector estimates 17 COCO joints per visible person; ByteTrack associates detections over time. This stage has no anomaly detector. Pixel x/y and confidence are saved for each joint, person track and source frame.

Every cached export is checked for finite poses, frame-ID bounds, full preview decoding, frame count, dimensions and source FPS. A malformed export raises an error rather than being treated as a successful run.

In [ ]:
from scripts.extract_video_poses import extract, verify_export
from src.demo.video_pose import DEFAULT_YOLO_CHECKPOINT

RUNS = {
    "Shoplifting baseline (conf 0.3)": (SOURCES["Shoplifting (dataset label)"], ROOT / "outputs/poses/Shoplifting033_x264", 0.3),
    "Normal baseline (conf 0.3)": (SOURCES["Normal (dataset label)"], ROOT / "outputs/poses/Normal_Videos_100_x264", 0.3),
    "Shoplifting comparison (conf 0.5)": (SOURCES["Shoplifting (dataset label)"], ROOT / "outputs/poses/Shoplifting033_conf05", 0.5),
}
verified = {}
for label, (source, folder, threshold) in RUNS.items():
    if not (folder / "summary.json").exists():
        if RUN_POSE_INFERENCE:
            extract(source, folder, DEFAULT_YOLO_CHECKPOINT, threshold)
        else:
            print(f"PENDING: {label}. Set RUN_POSE_INFERENCE=True to create its export.")
            continue
    verified[label] = verify_export(folder)
assert len(verified) == len(RUNS), "Create the missing pose exports, then rerun from this cell."
summary_table = pd.DataFrame([
    {"run": label, "source_frames": s["source_frames"], "FPS": s["fps"], "tracks": s["tracks"],
     "tracked_frame_coverage": round(s["track_frame_coverage"], 4),
     "windows_all_tracks": s["preprocessing_windows"]}
    for label, s in verified.items()
])
display(summary_table)
print("CHECKPOINT PASS: all displayed exports were verified in this notebook session.")

## 5 · Visual quality gate: inspect false detections and IDs
automated validity is not detection accuracy. In the baseline clothing-store preview, hanging clothes can create a false person skeleton. Compare detection thresholds using identical source footage. A lower track count alone does not establish improved accuracy.

**Manual check:** do limbs follow visible people? Does the main ID stay stable? Are clothing displays mistaken for people? Inspect the entire saved videos after this short analysis excerpt. Camera cuts, occlusions and partial bodies can cause new IDs. Track count is not unique-person count.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (label, (_, folder, _)) in zip(axes, RUNS.items()):
    ax.imshow(cv2.cvtColor(cv2.imread(str(folder / "preview.jpg")), cv2.COLOR_BGR2RGB))
    ax.set_title(label, fontsize=10); ax.axis("off")
plt.tight_layout(); plt.show()
for label in ["Shoplifting comparison (conf 0.5)", "Normal baseline (conf 0.3)"]:
    _, folder, _ = RUNS[label]
    display(Markdown(f"**{label} — pose IDs only, no theft predictions**"))
    show_clip(folder / "skeletons.mp4", start=20 if "Shoplifting" in label else 0, seconds=8)
quality_notes = [
    "Manual full-video inspection is still required; valid files are not an accuracy benchmark.",
    "Confidence 0.5 is a development comparison, not a globally validated operating threshold.",
    "Normal footage contains camera changes; expect track discontinuities at cuts.",
]
for note in quality_notes:
    print("REVIEW:", note)

## 6 · Inspect saved joints and track continuity
each observation has shape `(17, 3)` with pixel x, pixel y and confidence. Inspect the longest track to avoid confusing short false detections with the principal person. The example below is selected by track length, not by a manually supplied theft label.

In [ ]:
from src.data.retails import PersonTrack, PoseFrame
from src.demo.render import COCO17_BONES
from src.data.preprocessing import track_to_array, preprocess_track, build_windows

selected_label = "Shoplifting comparison (conf 0.5)"
source, pose_folder, _ = RUNS[selected_label]
pose_records = json.loads((pose_folder / "poses.json").read_text())
lengths = sorted(((tid, len(frames)) for tid, frames in pose_records.items()), key=lambda p: p[1], reverse=True)
display(pd.DataFrame(lengths, columns=["track_id", "observations"]))
track_id = lengths[0][0]
track = PersonTrack(person_id=track_id, camera_id="external_video", video_id=source.stem,
                    part="1", source_file=source,
                    frames={int(fid): PoseFrame(int(fid), np.array(record["keypoints"], dtype=np.float32).reshape(17, 3), None)
                            for fid, record in pose_records[track_id].items()})
frame_ids, raw_points = track_to_array(track)
assert raw_points.shape == (len(frame_ids), 17, 3) and np.isfinite(raw_points).all()
visible_joints = (raw_points[:, :, 2] >= 0.3).sum(axis=1)
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(frame_ids / verified[selected_label]["fps"], visible_joints, color="#117b78")
ax.set(xlabel="Source time (seconds)", ylabel="Confident joints", ylim=(0, 18), title=f"Track {track_id}: joint visibility at confidence ≥ 0.3")
plt.tight_layout(); plt.show()
print("CHECKPOINT PASS:", raw_points.shape, "finite values; frame IDs retain source-video indexing.")

## 7 · Preprocess and build temporal windows
mask low-confidence joints, interpolate only short gaps, smooth motion, center at the pelvis, scale by torso length, then build contiguous windows. Missing joints retain a visibility mask.

This inspection export covers the **longest track only**, not the complete training dataset. It has no manufactured frame-level anomaly labels. Saved windows and source provenance can be inspected independently.

Smoothing is isolated to contiguous frame-ID segments, so a pose after a tracking gap cannot alter the pose before that gap. This is covered by a regression test.

In [ ]:
cfg = yaml.safe_load((ROOT / "configs/preprocessing.yaml").read_text())
processed = preprocess_track(track, confidence_threshold=cfg["confidence_threshold"],
    max_interpolation_gap=cfg["max_interpolation_gap"], smoothing_window=cfg["smoothing_window"])
windows = build_windows(processed["frame_ids"], processed["keypoints"], processed["visibility_mask"],
                        window_length=cfg["window_length"], stride=cfg["stride"])
assert windows, "No contiguous windows: inspect track length and frame gaps."
window_points = np.stack([w["keypoints"] for w in windows])
window_masks = np.stack([w["visibility_mask"] for w in windows])
starts = np.array([w["start_frame"] for w in windows])
ends = np.array([w["end_frame"] for w in windows])
assert window_points.shape[1:] == (cfg["window_length"], 17, 3)
assert window_masks.shape == window_points.shape[:3] and np.isfinite(window_points).all()
assert np.all(ends - starts == cfg["window_length"] - 1), "A window bridged a tracking gap."

index = len(frame_ids) // 2
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, points, mask, title in [
    (axes[0], raw_points[index], raw_points[index, :, 2] >= cfg["confidence_threshold"], "Raw pixel-coordinate pose"),
    (axes[1], processed["keypoints"][index], processed["visibility_mask"][index], "Cleaned and normalized pose")]:
    for a, b in COCO17_BONES:
        if mask[a] and mask[b]: ax.plot(points[[a,b], 0], points[[a,b], 1], color="#117b78")
    ax.scatter(points[mask, 0], points[mask, 1], s=20, color="#e68a2e")
    ax.invert_yaxis(); ax.set_aspect("equal"); ax.set_title(title)
plt.tight_layout(); plt.show()
window_file = OUT / f"{pose_folder.name}_longest_track_windows.npz"
np.savez_compressed(window_file, keypoints=window_points, visibility_mask=window_masks,
                    start_frames=starts, end_frames=ends, source_video=str(source.resolve()),
                    source_sha256=hashlib.sha256(source.read_bytes()).hexdigest(),
                    track_id=track_id, config_json=json.dumps(cfg))
with np.load(window_file, allow_pickle=False) as saved:
    np.testing.assert_array_equal(saved["keypoints"], window_points)
    np.testing.assert_array_equal(saved["visibility_mask"], window_masks)
print("CHECKPOINT PASS:", window_points.shape, "window tensor; masks, continuity and saved round-trip verified.")
print("Saved:", window_file)

## 8 · Construct the existing 2D baseline features
Experiment A uses normalized x/y joint positions. Experiment B adds velocity for the selected motion joints. Both call the existing feature functions, using the same window tensors.

Shapes and finite values can be tested now. This does not train a model or measure anomaly-detection accuracy.

In [ ]:
from src.features.motion_features import build_feature_vector, feature_dim
features_a = np.stack([build_feature_vector(w[:, :, :2]) for w in window_points])
features_b = np.stack([build_feature_vector(w[:, :, :2], use_velocity=True) for w in window_points])
assert features_a.shape == (len(windows), cfg["window_length"], feature_dim())
assert features_b.shape == (len(windows), cfg["window_length"], feature_dim(use_velocity=True))
assert np.isfinite(features_a).all() and np.isfinite(features_b).all()
np.savez_compressed(OUT / "features_2d.npz", pose_only=features_a, pose_velocity=features_b,
                    source_windows=str(window_file), start_frames=starts, end_frames=ends, track_id=track_id)
display(pd.DataFrame([{"experiment": "A: position", "tensor_shape": str(features_a.shape)},
                      {"experiment": "B: position + velocity", "tensor_shape": str(features_b.shape)}]))
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(np.arange(cfg["window_length"]), features_a[0, :, 9*2], label="Left wrist normalized x")
ax.plot(np.arange(cfg["window_length"]), features_a[0, :, 10*2], label="Right wrist normalized x")
ax.set(xlabel="Frame within first window", ylabel="Normalized position", title="Example feature traces; review confidence masks alongside positions")
ax.legend(); plt.tight_layout(); plt.show()
print("CHECKPOINT PASS: finite feature tensors with the existing A/B feature dimensions.")

## 8b · Batch exports for all seven retail videos
the previous cells inspect one longest track. This phase exports every technically retained window from all seven videos using one detection setting (0.5), the shared preprocessing configuration, and explicit visibility filters.

Each window preserves source-video identity and hash, track ID, exact frame IDs, visibility mask, normalized poses, and aligned A/B features. Files are verified with checksums, dimensions, finite values, continuity and feature alignment. Low-visibility windows or unreliable torso anchors are rejected; these filters do **not** certify person identity or theft.

**Manual review remains pending.** The Normal/Shoplifting field describes the source video only. No per-window theft labels or train/test split are manufactured. Start with the saved exports for a quick analysis; set `RUN_BATCH_EXPORT=True` only to generate missing pose data and exports.

In [ ]:
batch_root = ROOT / "outputs/external_sequences/conf05"
if RUN_BATCH_EXPORT:
    run_cli("scripts/run_external_preprocessing.py", "--extract-missing")
if (batch_root / "batch_report.json").exists():
    run_cli("scripts/run_external_preprocessing.py", "--verify-only")
    batch_report = json.loads((batch_root / "batch_report.json").read_text())
    batch_table = pd.DataFrame(batch_report["videos"])
    display(batch_table[["video", "video_label", "tracks", "track_frame_coverage", "candidate_windows", "retained_windows", "rejected_windows", "manual_review_status"]])
    low_coverage = batch_table[batch_table["track_frame_coverage"] < 0.5]
    if not low_coverage.empty:
        print("QUALITY REVIEW REQUIRED: fewer than half the frames have tracked poses in", low_coverage["video"].tolist())
    assert batch_report["video_count"] == len(inventory), "Batch does not cover the current video inventory"
    assert not batch_report["training_ready"], "External-inference workflow data must not be presented as training-ready"
    ax = batch_table.set_index("video")[["retained_windows", "rejected_windows"]].plot.barh(
        stacked=True, color=["#117b78", "#e68a2e"], figsize=(10, 4))
    ax.set(xlabel="30-frame temporal windows", ylabel="", title="Technical visibility filtering — manual review still required")
    plt.tight_layout(); plt.show()
    sample_folder = Path(batch_report["videos"][0]["export_dir"])
    with np.load(sample_folder / "windows.npz", allow_pickle=False) as saved:
        print("Example export:", {k: saved[k].shape for k in ["keypoints", "visibility_mask", "frame_ids", "track_ids", "features_a", "features_b"]})
    print("CHECKPOINT PASS:", batch_report["video_count"], "videos;", batch_report["retained_windows"], "retained windows with verified provenance.")
    print("Video labels only; manual track review pending; this is not a benchmark or training dataset.")
else:
    print("PENDING: run scripts/run_external_preprocessing.py --extract-missing, or enable RUN_BATCH_EXPORT.")

## 8c · Improve weak overhead views and review the trade-offs
the uniform nano/confidence-0.5 baseline had low coverage in two overhead clips. Compare it with a lower nano threshold and a stronger YOLOv8s model at confidence 0.15, both at 640-pixel inference resolution. These are development experiments on the same footage, not precision/recall measurements.

The small-model profiles are saved separately in `configs/external_video_profiles.yaml`; the original 340-window baseline stays unchanged. More coverage or more windows does not prove better identity tracking. Sampled overlays were reviewed; occlusion, cropped bodies and ID changes remain limitations.

In [ ]:
tracking_comparison_path = ROOT / "outputs/tracking_review/comparison.json"
profile_root = ROOT / "outputs/external_sequences/candidate_profiles"
if tracking_comparison_path.exists() and (profile_root / "batch_report.json").exists():
    run_cli("scripts/run_external_preprocessing.py", "--profiles", "configs/external_video_profiles.yaml", "--out-root", profile_root, "--verify-only")
    tracking_comparison = pd.DataFrame(json.loads(tracking_comparison_path.read_text()))
    display(tracking_comparison[["video", "variant", "coverage", "tracks", "longest_track_frames", "retained_windows"]])
    for name in ["Shoplifting017_x264", "Shoplifting031_x264"]:
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        for ax, variant in zip(axes, ["baseline", "small_conf015_640"]):
            row = tracking_comparison[(tracking_comparison.video == name) & (tracking_comparison.variant == variant)].iloc[0]
            pose_dir = Path(row.pose_dir)
            pose_summary = json.loads((pose_dir / "summary.json").read_text())
            review_points = json.loads((pose_dir / "poses.json").read_text())
            same_frame_id = 150
            frame = cv2.cvtColor(read_frame(Path(pose_summary["source_video"]), same_frame_id), cv2.COLOR_RGB2BGR)
            from src.demo.render import draw_skeleton, draw_label
            for tid, observations in review_points.items():
                if str(same_frame_id) in observations:
                    joints = np.array(observations[str(same_frame_id)]["keypoints"]).reshape(17, 3)
                    draw_skeleton(frame, joints, (255, 200, 0), 0.3)
                    draw_label(frame, joints, f"ID {tid}", (255, 200, 0))
            ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            ax.set_title(f"{variant}: {row.coverage:.0%} frame coverage"); ax.axis("off")
        fig.suptitle(name + " | identical source frame 150, joint threshold 0.3"); plt.tight_layout(); plt.show()
    display(Markdown("**Example improved overhead skeleton excerpt — pose IDs only**"))
    show_clip(ROOT / "outputs/tracking_review/small_conf015_640/Shoplifting031_x264/skeletons.mp4", start=2, seconds=8)
    candidate_report = json.loads((profile_root / "batch_report.json").read_text())
    print("CHECKPOINT PASS:", candidate_report["retained_windows"], "candidate-profile windows verified; original baseline preserved.")
else:
    print("PENDING: run the tracking comparison/profile export described in DEVELOPMENT_CHECKLIST.md.")

## 9 · BiGRU autoencoder architecture
the BiGRU autoencoder learns to reconstruct normal pose sequences. A reconstruction error becomes useful only after training and threshold calibration. The model below is **untrained architecture inspection only**. No random-weight scores or synthetic accuracy values are displayed.

In [ ]:
from src.models.bigru_autoencoder import BiGRUAutoencoder
model_cfg = yaml.safe_load((ROOT / "configs/experiment_a.yaml").read_text())
architecture = BiGRUAutoencoder(input_dim=feature_dim(), hidden_size=model_cfg["hidden_size"],
    num_layers=model_cfg["num_layers"], dropout=model_cfg["dropout"])
print(architecture)
print("Parameters:", sum(p.numel() for p in architecture.parameters()))
print("NOT TRAINED: no anomaly scores or detection metrics have been computed in this cell.")

## 10 · Training and 3D asset checks
RetailS provides the training and held-out test pose sequences. The seven UCF videos support external-video inspection. MotionBERT code and a compatible pretrained checkpoint are required before 3D experiments can run.

Training uses the repository scripts and configurations. Keep source-video groups disjoint, hold out test data, and calibrate thresholds using normal data without consulting test labels.


In [ ]:
from src.data.retails import discover_retails_root, SUBSET_DIRS
retails_root = discover_retails_root(ROOT / "data/retails")
retails_ready = retails_root is not None and all(
    any((retails_root / folder / "pose").rglob("*.json")) for folder in SUBSET_DIRS.values())
mb_code = ROOT / "third_party/MotionBERT/lib/model/DSTformer.py"
mb_weights = ROOT / "third_party/MotionBERT/checkpoint/pose3d/MB_ft_h36m/best_epoch.bin"
mb_ready = mb_code.is_file() and mb_weights.is_file()
prerequisites = pd.DataFrame([
    {"stage": "RetailS train/test pose data", "available": retails_ready},
    {"stage": "MotionBERT code + compatible checkpoint", "available": mb_ready},
    {"stage": "Research D checkpoint + validation calibration", "available": all((ROOT / "outputs/checkpoints/research_v1/matched_d_seed0" / f).is_file() for f in ["model.pt", "validation_errors.npy"])},
])
display(prerequisites)
if RUN_RETAILS_EDA and retails_ready:
    run_cli("scripts/run_eda.py")
    run_cli("scripts/run_preprocessing.py", "--subset", "train", "--limit", "5")
else:
    print("RetailS EDA: not executed (disabled or data missing).")
assert RESEARCH_STAGE in ["matched", "expanded", "all"]
if RUN_TRAINING and retails_ready and mb_ready:
    run_cli("scripts/run_research_phase.py", "--stage", RESEARCH_STAGE)
else:
    print("Research training: disabled; saved results are inspected below.")

if RUN_3D_LIFT and mb_ready:
    from src.motionbert.lift import load_motionbert, lift_2d_to_3d
    from src.data.quality import prepare_track, accepted_window
    research_cfg = yaml.safe_load((ROOT / "configs/research_v1.yaml").read_text())
    pixel_processed = prepare_track(track, research_cfg, normalize=False, coordinate_order="xy")
    candidates = build_windows(pixel_processed["frame_ids"], pixel_processed["keypoints"], pixel_processed["visibility_mask"],
                               window_length=research_cfg["window_length"], stride=research_cfg["stride"])
    pixel_windows = []
    for candidate in candidates:
        start = int(np.searchsorted(pixel_processed["frame_ids"], candidate["start_frame"]))
        if accepted_window(candidate, pixel_processed["anchor_valid"][start:start + research_cfg["window_length"]], research_cfg):
            pixel_windows.append(candidate)
    assert pixel_windows, "No quality-supported windows for this track."
    lifted = lift_2d_to_3d(load_motionbert(mb_weights), pixel_windows[0]["keypoints"])
    assert lifted.shape == (cfg["window_length"], 17, 3) and np.isfinite(lifted).all()
    fig = plt.figure(figsize=(5, 4)); ax = fig.add_subplot(111, projection="3d")
    ax.scatter(lifted[0,:,0], lifted[0,:,1], lifted[0,:,2]); ax.set_title("Actual MotionBERT output: first pose")
    plt.show()
else:
    print("3D lifting: not executed (disabled or pretrained assets missing).")
if RUN_VIDEO_INFERENCE and mb_ready and bool(prerequisites.iloc[2]["available"]):
    run_cli("scripts/run_video_inference.py")
    show_clip(ROOT / "outputs/video_inference/matched_d_seed0/Shoplifting033_x264/annotated.mp4", start=20)
else:
    print("Full scored inference workflow: not executed (disabled or trained assets missing).")

## 10b · Reference: original A training and evaluation
the downloaded RetailS pose dataset is the research training/evaluation source. The UCF clips remain external development footage. The verification command checks all test-mask associations/binary values and samples five pose schemas per subset; it does not claim a full coordinate audit of every pose.

The first Experiment A run uses 50 sampled normal-training pose files, the existing architecture and 15 epochs. Training/validation windows are grouped by camera and source video, keeping parts of the same video together. The actual validation fraction of **windows** can differ from the requested fraction of **video groups**. Test subsets are held out, and the decision threshold uses normal-training reconstruction errors only. Results below load saved, measured outputs; the cell does not silently retrain the model.

In [ ]:
measured_metrics = None
retails_audit = None
if retails_ready:
    run_cli("scripts/verify_retails.py")
    retails_audit = json.loads((ROOT / "outputs/logs/retails_verification.json").read_text())
    display(pd.DataFrame(retails_audit["subsets"]).T)
else:
    print("PENDING: RetailS dataset is unavailable.")
training_log = ROOT / "outputs/logs/experiment_a/training_log.csv"
split_file = ROOT / "outputs/logs/experiment_a/split_manifest.json"
metrics_file = ROOT / "outputs/logs/experiment_a/eval_metrics.json"
if training_log.exists() and split_file.exists() and metrics_file.exists():
    history = pd.read_csv(training_log)
    split = json.loads(split_file.read_text())
    assert not (set(map(tuple, split["train_groups"])) & set(map(tuple, split["validation_groups"])))
    assert np.isfinite(history[["train_loss", "val_loss"]].to_numpy()).all()
    display(pd.DataFrame([{"split_method": split["method"], "train_video_groups": len(split["train_groups"]),
                          "validation_video_groups": len(split["validation_groups"]), "train_windows": split["train_windows"],
                          "validation_windows": split["validation_windows"], "epochs": len(history),
                          "validation_window_fraction": split["actual_validation_window_fraction"]}]))
    ax = history.plot(x="epoch", y=["train_loss", "val_loss"], figsize=(9, 3))
    ax.set(ylabel="Reconstruction MSE", title="Experiment A — measured learning curves, disjoint source-video groups")
    plt.tight_layout(); plt.show()
    measured_metrics = json.loads(metrics_file.read_text())
    evaluation_table = pd.DataFrame(measured_metrics).T
    evaluation_table["anomaly_window_fraction"] = evaluation_table.num_anomaly / (evaluation_table.num_anomaly + evaluation_table.num_normal)
    display(evaluation_table[["num_normal", "num_anomaly", "anomaly_window_fraction", "precision", "recall", "f1", "specificity", "roc_auc", "pr_auc"]])
    combined = measured_metrics["combined"]
    print(f"Measured baseline: precision {combined['precision']:.1%}, recall {combined['recall']:.1%}, F1 {combined['f1']:.3f}, ROC-AUC {combined['roc_auc']:.3f}.")
    if combined["recall"] < 0.5:
        print("LIMITATION: this operating point misses most anomalous windows. A high precision value does not establish a useful detector.")
    print("Class balance matters: PR-AUC must be interpreted alongside the anomaly-window prevalence. Window labels use ANY anomalous video frame; they are not person-specific theft annotations.")
    assert set(measured_metrics) == {"test_realworld", "test_staged", "combined"}
    errors = np.load(ROOT / "outputs/checkpoints/experiment_a/train_reconstruction_errors.npy", allow_pickle=False)
    assert np.isfinite(errors).all()
    print("CHECKPOINT PASS: grouped split is disjoint; actual training curves and held-out evaluation loaded.")
    print("Interpret these as one sampled-data 2D baseline; poor results or false alarms are findings, not hidden failures.")
    print("These are the original reference results. The quality-corrected research runs follow in sections 10d–10f.")
else:
    print("PENDING: Experiment A training/evaluation outputs are not yet complete; no scores invented.")

## 10c · Reference: original A/B comparison
Experiment A uses 34 position features; B uses 58 position-plus-velocity features. Both use 50 sampled normal pose files, the same disjoint source-video split, 15 epochs and final-epoch checkpoints. Thresholds are calibrated independently from each model's saved normal reconstruction errors at p95. No test labels are used to adjust thresholds.

The report verifies matching window identities and labels before comparing metrics. Confusion counts and the anomaly prevalence provide context for high precision or average precision. Error rows retain the source camera, pose file, person and frame interval; they do not imply person-specific theft annotations.


In [ ]:
ab_report = None
ab_path = ROOT / "outputs/logs/ab_comparison/comparison.json"
if ab_path.is_file():
    ab_report = json.loads(ab_path.read_text())
    assert ab_report["matched_training_split"] and ab_report["matched_test_windows"]
    comparison_rows = []
    fig, axes = plt.subplots(1, 2, figsize=(12, 3))
    for ax, (name, result) in zip(axes, ab_report["experiments"].items()):
        checkpoint_file = ROOT / f"outputs/checkpoints/experiment_{name}/model.pt"
        assert hashlib.sha256(checkpoint_file.read_bytes()).hexdigest() == result["checkpoint_sha256"], "Comparison is stale; rerun compare_2d_experiments.py."
        for subset, metrics in {**result["subsets"], "combined": result["combined"]}.items():
            comparison_rows.append({"experiment": name.upper(), "subset": subset,
                **{k: metrics[k] for k in ["num_normal", "num_anomaly", "anomaly_fraction", "threshold", "precision", "recall", "f1", "roc_auc", "pr_auc"]},
                **metrics["confusion"]})
        history = pd.read_csv(ROOT / f"outputs/logs/experiment_{name}/training_log.csv")
        history.plot(x="epoch", y=["train_loss", "val_loss"], ax=ax)
        ax.set(title=f"Experiment {name.upper()}: {result['feature_dim']} features", ylabel="Reconstruction MSE")
    plt.tight_layout(); plt.show()
    display(pd.DataFrame(comparison_rows))
    display(Markdown("Loss scales depend on feature composition; use held-out detection metrics to compare models."))
    display(Markdown(ab_report["threshold_protocol"]))
    image = cv2.imread(str(ab_path.parent / "comparison.png"))
    assert image is not None
    plt.figure(figsize=(12, 10)); plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.show()
    predictions = pd.read_csv(ab_path.parent / "window_predictions.csv")
    identity = ["subset", "pose_file", "person_id", "camera_id", "video_id", "start_frame", "end_frame"]
    a_rows = predictions[predictions.experiment == "a"].set_index(identity).sort_index()
    b_rows = predictions[predictions.experiment == "b"].set_index(identity).sort_index()
    assert a_rows.index.is_unique and a_rows.index.equals(b_rows.index)
    assert np.array_equal(a_rows.label.to_numpy(), b_rows.label.to_numpy())
    assert len(a_rows) == ab_report["test_windows"]
    print("Prediction changes on identical windows:")
    display(pd.crosstab(a_rows.outcome, b_rows.outcome, rownames=["A outcome"], colnames=["B outcome"]))
    print("Missed anomaly windows by camera:")
    display(predictions[predictions.outcome == "FN"].groupby(["experiment", "subset", "camera_id"]).size().rename("missed_windows").reset_index())
    print("Lowest-scoring missed anomaly examples (source frame indices):")
    display(predictions[predictions.outcome == "FN"].sort_values(["experiment", "score"]).groupby("experiment").head(5))
    for name, result in ab_report["experiments"].items():
        print(f"Experiment {name.upper()}: saved normal calibration and held-out score distributions")
        display(pd.DataFrame([result["calibration_score_quantiles"]], index=["normal calibration"]))
        display(pd.DataFrame(result["score_quantiles"]).T)
        for relative_path in result["error_pose_plots"]:
            if "realworld_fn" in relative_path:
                image = cv2.imread(str(ROOT / relative_path))
                assert image is not None
                plt.figure(figsize=(12, 3.5)); plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.show()
    print("Pose strips show normalized joints for missed video-mask windows; they do not prove which person committed theft.")
    print("CHECKPOINT PASS: saved checkpoint hashes verified; A/B use identical test windows and labels.")
    print("LIMITATION: one seed and 50 sampled pose files do not establish generalization. Review recall and ROC-AUC before expanding experiments.")
else:
    print("PENDING: train/evaluate A and B, then run scripts/compare_2d_experiments.py.")


## 10d · Input-quality audit and validated 3D inputs
The original pipeline retained missing-joint coordinates in smoothing, normalization and loss. The new pipeline sanitizes those coordinates, smooths only supported observations, requires usable torso anchors, and applies a visibility-aware reconstruction loss.

RetailS uses an explicit `yx` interpretation in the new configuration: body geometry across six cameras and the authors' upright examples support this conversion, while the source documentation calls the records XYC. Source RGB/calibration is unavailable. YOLO video inputs retain their existing `xy` order.

MotionBERT validation checks joint indices, parity with official normalization, frozen weights, finite output and a zero root. Plausible plots do not establish 3D accuracy without ground truth.


In [ ]:
quality_audit = None
research_input_validation = None
audit_path = ROOT / "outputs/logs/pose_quality_audit/audit.json"
validation_path = ROOT / "outputs/logs/research_input_validation.json"
if audit_path.is_file():
    quality_audit = json.loads(audit_path.read_text())
    display(pd.DataFrame(quality_audit["subsets"]).T)
    display(pd.DataFrame(quality_audit["worst_normal_windows"]).head(5))
    print("Audit scope:", quality_audit["scope"])
if validation_path.is_file():
    research_input_validation = json.loads(validation_path.read_text())
    assert all(research_input_validation[k] for k in ["official_normalization_parity", "direct_joint_mapping_exact", "frozen_model", "output_finite", "root_relative"])
    assert hashlib.sha256(mb_weights.read_bytes()).hexdigest() == research_input_validation["checkpoint_sha256"]
    display(pd.DataFrame(research_input_validation["examples"]))
    for example in research_input_validation["examples"]:
        if example["camera_id"] in ["1", "3", "5"]:
            image = cv2.imread(str(ROOT / example["plot"]))
            assert image is not None
            plt.figure(figsize=(15, 5)); plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.show()
    print("CHECKPOINT PASS: recorded structural checks and checkpoint hash verified.")
    print(research_input_validation["limitation"])
    print(research_input_validation["visual_review_status"])
    for note in research_input_validation["visual_review_notes"]:
        print("Review:", note)
else:
    print("PENDING: run scripts/prepare_motionbert.py and scripts/validate_research_inputs.py.")


## 10e · Matched A–E experiments and expanded validation
The matched experiment uses the same 10 normal-training pose files, source-video split and retained test windows for all five models. A/B use 2D features; C/D use frozen 3D lifting; E adds temporal attention to D. Hidden width/depth and the 15-epoch budget are fixed; projection parameter counts vary with input dimension.

The expanded A family was declared before test evaluation: 100 training files and model seeds 0/1/2 with a fixed source-video split. Checkpoints minimize normal validation loss; thresholds use p95 of normal validation reconstruction errors. Test labels select neither.

Report retained-window metrics alongside **candidate-window recall**, which counts rejected positive windows as misses. Rejected windows are unscored, not normal. The frame masks still do not identify the responsible person.


In [ ]:
from src.experiments.artifacts import run_paths
research_report = None
research_path = ROOT / "outputs/logs/research_v1/final_report.json"
if research_path.is_file():
    research_report = json.loads(research_path.read_text())
    if research_report["complete"]:
        assert research_report["matched_splits_identical"]
        for name, result in research_report["runs"].items():
            checkpoint = run_paths(name)[0] / "model.pt"
            assert hashlib.sha256(checkpoint.read_bytes()).hexdigest() == result["checkpoint_sha256"]
        table = pd.DataFrame(research_report["comparisons"])
        display(table[["run", "training_files", "train_windows", "validation_windows", "best_epoch", "feature_dim"]])
        display(table[["run", "test_windows", "unscored_candidate_windows", "precision", "recall", "candidate_window_recall", "f1", "roc_auc", "pr_auc"]])
        display(pd.DataFrame(research_report["expanded_three_seed_summary"]).T)
        if research_report.get("expanded_b_three_seed_summary"):
            print("Expanded B: mean and sample standard deviation")
            display(pd.DataFrame(research_report["expanded_b_three_seed_summary"]).T)
        print("Original A/B evaluated on the same retained-window intersection:")
        display(pd.DataFrame(research_report["legacy_on_retained_windows"]))
        for plot in research_report["plots"]:
            image = cv2.imread(str(ROOT / plot))
            assert image is not None
            plt.figure(figsize=(15, 5)); plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.show()
        camera_metrics = pd.read_csv(research_path.parent / "per_camera_metrics.csv")
        display(camera_metrics[camera_metrics.run.isin(["matched_d_seed0", "expanded_a_seed0", "expanded_a_seed1", "expanded_a_seed2", "expanded_b_seed0", "expanded_b_seed1", "expanded_b_seed2"])])
        display(pd.DataFrame(research_report["run_artifacts"]).T)
        display(Markdown("### Secondary frame-level diagnostic"))
        frame_diagnostic = research_report["frame_level_diagnostic"]
        frame_table = pd.DataFrame([{"run": name, **r["metrics"]["combined"]} for name, r in frame_diagnostic["runs"].items()])
        display(frame_table[["run", "score_coverage", "precision", "recall", "all_annotated_anomaly_recall", "f1", "roc_auc", "pr_auc"]])
        print(frame_diagnostic["protocol"])
        print("This secondary diagnostic does not replace the predeclared window results. Unscored frames remain unknown.")
        print("Frame thresholds differ because max aggregation changes the normal validation score distribution.")
        for finding in research_report["findings"]:
            print("Finding:", finding)
        print("Traceable failure examples from the corrected models:")
        display(pd.DataFrame(research_report["error_examples"]))
        for example in research_report["error_examples"]:
            if example["run"] in ["matched_d_seed0", "expanded_a_seed0", "expanded_b_seed0"] and example["subset"] == "test_realworld":
                image = cv2.imread(str(ROOT / example["plot"]))
                assert image is not None
                plt.figure(figsize=(12, 4)); plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.show()
        print(f"CHECKPOINT PASS: {len(research_report['runs'])} trained runs, source-video splits and checkpoint hashes verified.")
        print("Interpret AP alongside anomaly prevalence; use recall and abstention coverage when judging usefulness.")

        analysis_path = research_path.parent / "error_analysis/report.json"
        if analysis_path.exists():
            analysis = json.loads(analysis_path.read_text())
            print("Expanded model error analysis:")
            for finding in analysis["findings"]:
                print(finding)
            display(pd.read_csv(analysis_path.parent / "frame_camera_summary.csv"))
            for relative in analysis["plots"]:
                image = cv2.imread(str(ROOT / relative))
                assert image is not None
                plt.figure(figsize=(14, 7)); plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.show()
            print("Next experiment:", analysis["next_experiment"])

        ab_expanded_path = research_path.parent / "expanded_ab_comparison/comparison.json"
        if ab_expanded_path.exists():
            expanded_ab = json.loads(ab_expanded_path.read_text())
            for name, expected in expanded_ab["checkpoint_sha256"].items():
                assert hashlib.sha256((run_paths(name)[0] / "model.pt").read_bytes()).hexdigest() == expected
            assert expanded_ab["windows_identical"] and expanded_ab["splits_and_configs_identical"]
            display(Markdown("### Expanded A versus B: positions and explicit motion"))
            display(pd.DataFrame(expanded_ab["means"]))
            for finding in expanded_ab["findings"]:
                print(finding)
            print("B CPU portability check:", expanded_ab["cpu_verified_windows"], "sampled windows passed")
            for plot in expanded_ab["plots"]:
                image = cv2.imread(str(ROOT / plot))
                assert image is not None
                plt.figure(figsize=(14, 8)); plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.show()
            print("Next step:", expanded_ab["next_step"])
    else:
        print("PENDING:", research_report["missing"])
else:
    print("PENDING: complete research runs and scripts/summarize_research_phase.py.")


## 10f · Offline inference on external retail videos
The predeclared D configuration scores the cached YOLO tracks using the same quality preparation, frozen lift and features as training. Input YOLO coordinates already use `xy`. Completed-window scores expire after one stride, so a later tracking gap cannot inherit an old result.

Inspect timestamps, gray unscored tracks, above-threshold scores and original source frames together. Full-track interpolation/smoothing is retrospective; this is not causal streaming. These videos have no person/interval annotations for measuring detection accuracy.


In [ ]:
video_verification = None
video_report_path = ROOT / "outputs/video_inference/matched_d_seed0/verification.json"
if video_report_path.is_file():
    video_verification = json.loads(video_report_path.read_text())
    assert len(video_verification) == 7
    for result in video_verification:
        assert result["source_frames"] == result["decoded_output_frames"]
        assert np.isclose(result["source_fps"], result["output_fps"])
    video_table = pd.DataFrame(video_verification)
    video_table["video"] = video_table["video"].map(lambda p: Path(p).name)
    display(video_table[["video", "candidate_windows", "retained_windows", "rejected_windows", "flagged_windows", "video_frame_score_coverage", "source_frames", "decoded_output_frames", "output_fps"]])
    for stem, start in [("Normal_Videos_100_x264", 3), ("Shoplifting033_x264", 20)]:
        folder = ROOT / "outputs/video_inference/matched_d_seed0" / stem
        show_clip(folder / "annotated.mp4", start=start)
    scores_path = ROOT / "outputs/video_inference/matched_d_seed0/Shoplifting033_x264/scores.json"
    scored = json.loads(scores_path.read_text())
    windows_table = pd.DataFrame(scored["windows"])
    if not windows_table.empty:
        longest_id = windows_table.track_id.value_counts().index[0]
        group = windows_table[windows_table.track_id == longest_id]
        plt.figure(figsize=(10, 3))
        plt.scatter(group.end_seconds, group.score, c=group.above_threshold.map({True: "red", False: "green"}), s=16)
        plt.axhline(scored["threshold"], color="black", linestyle="--", label="Validation-normal p95 threshold")
        plt.xlabel("Source window end time (seconds)"); plt.ylabel("Masked reconstruction error")
        plt.title(f"Shoplifting033: scored track {longest_id}; no person/event ground truth")
        plt.legend(); plt.tight_layout(); plt.show()
    print("CHECKPOINT PASS: all seven output videos were fully decoded with source FPS/frame counts preserved.")
else:
    print("PENDING: run scripts/run_video_inference.py after matched D training.")


## 10g. Alignment audit and autonomous retail review

The seven source timelines and skeleton samples have been reviewed by the assistant. Observations are explicitly AI-attributed and remain uncertain action evidence, not supervised labels or independent ground truth. The local report also compares frozen A/B scores before and after isolating verified camera zoom/pan intervals. No user annotation task is required.

In [ ]:
alignment_path = ROOT / "outputs/validation_alignment/report.json"
review_folder = ROOT / "outputs/manual_validation"
if alignment_path.exists():
    alignment = json.loads(alignment_path.read_text())
    display(pd.DataFrame([alignment["totals"]]))
    print("Structural alignment checks passed:", alignment["structural_checks_passed"])
    print("Semantic alignment:", alignment["semantic_alignment_status"])
    for finding in alignment["findings"]:
        print(finding)
if (review_folder / "preparation_report.json").exists():
    review_preparation = json.loads((review_folder / "preparation_report.json").read_text())
    display(pd.DataFrame([review_preparation]))
    print("Review page:", review_folder / "index.html")
    validations = sorted(review_folder.glob("annotations/*/validation.json"))
    if validations:
        display(pd.DataFrame([json.loads(p.read_text()) for p in validations]))
    else:
        print("No validated observation records available.")
    print("Existing clips are a pilot, not an independent confirmation set. Follow VALIDATION_REVIEW.md.")
pilot_path = ROOT / "outputs/retail_pilot_analysis/report.json"
if pilot_path.exists():
    pilot = json.loads(pilot_path.read_text())
    assert not pilot["accuracy_metrics_computed"] and pilot["reviewer_type"] == "ai_assisted"
    assert hashlib.sha256((ROOT / "configs/retail_camera_motion.json").read_bytes()).hexdigest() == pilot["camera_guard_sha256"]
    assert hashlib.sha256((review_folder / "ai_review.json").read_bytes()).hexdigest() == pilot["review_sha256"]
    for name, expected in pilot["checkpoint_sha256"].items():
        assert hashlib.sha256((run_paths(name)[0] / "model.pt").read_bytes()).hexdigest() == expected
    print("Autonomous review complete; no reference action labels were generated.")
    display(pd.DataFrame(pilot["pose_quality"]))
    pilot_rows = pd.DataFrame(pilot["comparisons"])
    display(pilot_rows[pilot_rows.case_id.isin(["RV002", "RV007"]) & pilot_rows.run.str.endswith("seed0")][["case_id", "run", "mode", "accepted_windows", "score_coverage", "flagged_windows"]])
    for finding in pilot["findings"]:
        print(finding)
    for relative in pilot["plots"]:
        image = cv2.imread(str(ROOT / relative))
        assert image is not None
        plt.figure(figsize=(14, 7)); plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.show()
    print("Read-only results:", review_folder / "index.html")

annotated_path = ROOT / "outputs/video_analysis/verification.json"
if annotated_path.exists():
    from IPython.display import Video
    annotated = json.loads(annotated_path.read_text())
    assert len(annotated) == 7
    for item in annotated:
        assert item["decoded_output_frames"] == item["source_frames"]
        assert item["output_fps"] == item["source_fps"]
        assert not item["action_confirmed"]
        assert hashlib.sha256((ROOT / item["video"]).read_bytes()).hexdigest() == item["video_sha256"]
    display(pd.DataFrame(annotated)[["case_id", "decoded_output_frames", "output_fps", "scored_frames", "unverified_alert_frames"]])
    print("Verified annotated frames:", sum(item["decoded_output_frames"] for item in annotated))
    print("Red banners are unverified model alerts, not confirmed theft. Scores are not probabilities.")
    for item in annotated:
        if item["case_id"] in ("RV002", "RV007"):
            print(item["case_id"], "- camera-aware annotated model output")
            display(Video(str(ROOT / item["video"]), embed=True, width=960))

failure_path = ROOT / "outputs/model_failure_audit/report.json"
if failure_path.exists():
    failure_audit = json.loads(failure_path.read_text())
    assert failure_audit["complete"] and not failure_audit["changes_to_models_or_thresholds"]
    assert failure_audit["totals"]["independently_verified_new_action_labels"] == 0
    for name, provenance in failure_audit["models"].items():
        model_folder, log_folder = run_paths(name)
        assert hashlib.sha256((model_folder / "model.pt").read_bytes()).hexdigest() == provenance["checkpoint_sha256"]
        assert hashlib.sha256((log_folder / "window_predictions.csv").read_bytes()).hexdigest() == provenance["predictions_sha256"]
    display(pd.DataFrame([failure_audit["totals"]]))
    for finding in failure_audit["findings"]:
        print(finding)
    failure_table = pd.read_csv(ROOT / "outputs/model_failure_audit/frozen_run_errors.csv")
    display(failure_table[failure_table.subset == "combined"])
    image = cv2.imread(str(ROOT / "outputs/model_failure_audit/failure_summary.png"))
    assert image is not None
    plt.figure(figsize=(14, 5)); plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.show()
    print("Independent action validation remains incomplete; no test-guided threshold tuning was performed.")


## 11 · Regression checks and reproducibility
Tests cover source FPS, missing inputs, malformed poses, tracking gaps, source-video splitting and error categorization. Model detection quality is assessed separately using held-out metrics and traceable error reports.


In [ ]:
run_cli("-m", "unittest", "discover", "-s", "tests", "-v")
report = {
    "python": sys.executable,
    "versions": {name: version(name) for name in ["torch", "ultralytics", "numpy", "opencv-python", "notebook"]},
    "input_videos": len(inventory),
    "batch_export": {"video_count": batch_report["video_count"], "retained_windows": batch_report["retained_windows"], "training_ready": batch_report["training_ready"]} if (batch_root / "batch_report.json").exists() else None,
    "verified_pose_runs": summary_table.to_dict(orient="records"),
    "longest_track_windows": len(windows),
    "window_shape": list(window_points.shape),
    "feature_shapes": {"a": list(features_a.shape), "b": list(features_b.shape)},
    "experiment_a_metrics": measured_metrics, "ab_comparison": ab_report, "retails_audit": retails_audit,
    "research_report": research_report, "quality_audit": quality_audit, "research_input_validation": research_input_validation, "video_verification": video_verification,
    "retails_ready": retails_ready, "motionbert_ready": mb_ready,
    "manual_review_notes": quality_notes,
    "run_controls": {"batch_export": RUN_BATCH_EXPORT, "frame_extraction": RUN_FRAME_EXTRACTION, "pose_inference_if_missing": RUN_POSE_INFERENCE,
                     "retails_eda": RUN_RETAILS_EDA, "training": RUN_TRAINING, "3d_lift": RUN_3D_LIFT, "video_inference": RUN_VIDEO_INFERENCE},
}
(OUT / "workflow_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print("CHECKPOINT PASS: regression tests passed; current-session evidence saved to", OUT / "workflow_report.json")
print("Next quality gate: review full overlays and profile comparisons before approving tracks.")
print("Research artifacts:", "complete" if research_report and research_report["complete"] else "pending stages listed above")

## Interpretation and reproducibility
- Pose quality and viewpoint affect every downstream model. Missing joints and low coverage remain explicit.
- MotionBERT output is inferred 3D structure without RetailS 3D ground truth.
- Video-mask window labels are not person-specific theft annotations.
- A–E share a matched cohort. Expanded A uses three seeds on a larger training sample and is reported separately.
- Conditional metrics can hide abstention; inspect candidate-window recall and rejected-window counts.
- External-video scores require qualitative review and temporal/person annotations before accuracy can be measured.
- Offline interpolation and smoothing do not establish real-time performance.

Restart the kernel and run all cells to verify reproducibility. Keep source notebooks free of outputs; generated videos, weights and arrays stay ignored by Git. See `MODEL_DEVELOPMENT.md` for the exact stage commands and remaining scientific limitations.
